In [ ]:
%%bash
for pilot in "Applications Test|apps-model-v1|applications" \
             "Healthcare Test|ns-2025-10-17|healthcare_oncology" \
             "Energy Test|grid-ai-v2|smart_energy"; do
  IFS='|' read name model tag <<< "$pilot"
  curl -s -X POST http://localhost:8000/api/v1/configuration/new \
    -H "Content-Type: application/json" \
    -d "{\"application_name\":\"$name\",\"ai_model_name\":\"$model\",
         \"ai_model_type\":\"classifier\",\"config_type\":\"haic_session\",
         \"pilot_tag\":\"$tag\"}" | python3 -c "import sys,json; print(json.load(sys.stdin)['id'])"
done

SyntaxError: unterminated string literal (detected at line 7) (1851325216.py, line 7)

In [ ]:
%%bash
# Replace 10, 11, 12 with your actual config IDs
curl -s -X POST "http://localhost:8000/api/v1/logs/upload?configuration_id=3" \
  -F "file=@test_applications_30sessions.json" | python3 -m json.tool

curl -s -X POST "http://localhost:8000/api/v1/logs/upload?configuration_id=4" \
  -F "file=@test_healthcare_20sessions.json" | python3 -m json.tool

curl -s -X POST "http://localhost:8000/api/v1/logs/upload?configuration_id=5" \
  -F "file=@test_energy_15sessions.json" | python3 -m json.tool

SyntaxError: invalid syntax (2801420511.py, line 2)

In [4]:
%%bash
for id in 3 4 5; do
  curl -s -X POST http://localhost:8000/api/v1/evaluate/$id | python3 -m json.tool
  sleep 2
done

{
    "detail": "Evaluation started for configuration 3",
    "message": "Evaluation started for configuration 3",
    "status": "running",
    "log_count": 1,
    "configuration_id": 3
}
{
    "detail": "Evaluation started for configuration 4",
    "message": "Evaluation started for configuration 4",
    "status": "running",
    "log_count": 1,
    "configuration_id": 4
}
{
    "detail": "Evaluation started for configuration 5",
    "message": "Evaluation started for configuration 5",
    "status": "running",
    "log_count": 1,
    "configuration_id": 5
}


In [1]:
%%bash
curl -s -X POST "http://localhost:8000/api/v1/logs/upload?configuration_id=3" \
  -F "file=@test_fairness_40sessions.json" | python3 -m json.tool

curl -s -X POST http://localhost:8000/api/v1/evaluate/3 | python3 -m json.tool

Expecting value: line 1 column 1 (char 0)


{
    "detail": "Evaluation started for configuration 3",
    "message": "Evaluation started for configuration 3",
    "status": "running",
    "log_count": 1,
    "configuration_id": 3
}


In [6]:
%%bash

python3 - << 'EOF'
import json, requests

with open("/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_data/test_fairness_40sessions_1.json") as f:
    data = json.load(f)

# Handle both list and wrapped dict format
if isinstance(data, dict):
    surveys = data.get("logs") or data.get("surveys") or list(data.values())[0]
else:
    surveys = data

ETHICS_MAP = {"e1": "q_fairness", "e2": "q_transparency",
              "e3": "q_privacy", "e4": "q_accountability", "e5": "q_trust"}

success = 0
for i, survey in enumerate(surveys):
    # Remap ethics keys
    survey["ethics_responses"] = {
        ETHICS_MAP.get(k, k): v
        for k, v in survey["ethics_responses"].items()
    }
    r = requests.post("http://localhost:8000/api/v1/survey", json=survey)
    if r.status_code == 200:
        success += 1
        print(f"Survey {i+1}: ✓")
    else:
        print(f"Survey {i+1}: {r.status_code} {r.text[:120]}")

print(f"\nSubmitted {success}/{len(surveys)} surveys successfully")
EOF

Traceback (most recent call last):
  File "<stdin>", line 20, in <module>
KeyError: 'ethics_responses'


CalledProcessError: Command 'b'\npython3 - << \'EOF\'\nimport json, requests\n\nwith open("/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_data/test_fairness_40sessions_1.json") as f:\n    data = json.load(f)\n\n# Handle both list and wrapped dict format\nif isinstance(data, dict):\n    surveys = data.get("logs") or data.get("surveys") or list(data.values())[0]\nelse:\n    surveys = data\n\nETHICS_MAP = {"e1": "q_fairness", "e2": "q_transparency",\n              "e3": "q_privacy", "e4": "q_accountability", "e5": "q_trust"}\n\nsuccess = 0\nfor i, survey in enumerate(surveys):\n    # Remap ethics keys\n    survey["ethics_responses"] = {\n        ETHICS_MAP.get(k, k): v\n        for k, v in survey["ethics_responses"].items()\n    }\n    r = requests.post("http://localhost:8000/api/v1/survey", json=survey)\n    if r.status_code == 200:\n        success += 1\n        print(f"Survey {i+1}: \xe2\x9c\x93")\n    else:\n        print(f"Survey {i+1}: {r.status_code} {r.text[:120]}")\

In [8]:
%%bash

python3 -c "
import json
with open('/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json') as f:
    d = json.load(f)
print(type(d))
if isinstance(d, dict):
    print('Keys:', list(d.keys()))
    print('First value type:', type(list(d.values())[0]))
elif isinstance(d, list):
    print('Length:', len(d))
    print('First item type:', type(d[0]))
    if isinstance(d[0], dict):
        print('First item keys:', list(d[0].keys())[:5])
"

Traceback (most recent call last):
  File "<string>", line 3, in <module>
FileNotFoundError: [Errno 2] No such file or directory: '/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json'


CalledProcessError: Command 'b'\npython3 -c "\nimport json\nwith open(\'/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json\') as f:\n    d = json.load(f)\nprint(type(d))\nif isinstance(d, dict):\n    print(\'Keys:\', list(d.keys()))\n    print(\'First value type:\', type(list(d.values())[0]))\nelif isinstance(d, list):\n    print(\'Length:\', len(d))\n    print(\'First item type:\', type(d[0]))\n    if isinstance(d[0], dict):\n        print(\'First item keys:\', list(d[0].keys())[:5])\n"\n'' returned non-zero exit status 1.

In [7]:
%%bash

python3 - << 'EOF'
import json, requests

with open("/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json") as f:
    surveys = json.load(f)

# Fix ethics field names to match actual schema
ETHICS_MAP = {"e1": "q_fairness", "e2": "q_transparency", 
              "e3": "q_privacy", "e4": "q_accountability", "e5": "q_trust"}

for i, survey in enumerate(surveys):
    # Remap ethics keys
    old_ethics = survey["ethics_responses"]
    survey["ethics_responses"] = {
        ETHICS_MAP[k]: v for k, v in old_ethics.items()
    }
    r = requests.post("http://localhost:8000/api/v1/survey", json=survey)
    print(f"Survey {i+1}: {r.status_code}", 
          r.json().get("detail", "ok") if r.status_code != 200 else "✓")

print("\nDone. Checking DB...")
EOF

Traceback (most recent call last):
  File "<stdin>", line 3, in <module>
FileNotFoundError: [Errno 2] No such file or directory: '/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json'


CalledProcessError: Command 'b'\npython3 - << \'EOF\'\nimport json, requests\n\nwith open("/Users/gfragi/postdoc/haic_benchmark_suite/haic_sim_mvp/test_datatest_sus_16surveys.json") as f:\n    surveys = json.load(f)\n\n# Fix ethics field names to match actual schema\nETHICS_MAP = {"e1": "q_fairness", "e2": "q_transparency", \n              "e3": "q_privacy", "e4": "q_accountability", "e5": "q_trust"}\n\nfor i, survey in enumerate(surveys):\n    # Remap ethics keys\n    old_ethics = survey["ethics_responses"]\n    survey["ethics_responses"] = {\n        ETHICS_MAP[k]: v for k, v in old_ethics.items()\n    }\n    r = requests.post("http://localhost:8000/api/v1/survey", json=survey)\n    print(f"Survey {i+1}: {r.status_code}", \n          r.json().get("detail", "ok") if r.status_code != 200 else "\xe2\x9c\x93")\n\nprint("\\nDone. Checking DB...")\nEOF\n'' returned non-zero exit status 1.

In [9]:
%%bash

docker compose exec backend python3 -c "
from app.utils.database import SessionLocal
from app.models.survey import Survey
db = SessionLocal()
total = db.query(Survey).count()
linked = db.query(Survey).filter(Survey.configuration_id == 3).count()
print(f'Total surveys: {total}')
print(f'Linked to config 3: {linked}')
db.close()
"

Total surveys: 16
Linked to config 3: 16
